In [1]:
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import json
import warnings
warnings.filterwarnings("ignore")

# Load dât
df = pd.read_parquet("../data/merged_crime_data.parquet")

print("Data loaded:", df.shape)
print(df[['latitude', 'longitude', 'city']].describe())

Data loaded: (696694, 10)
            latitude      longitude
count  696694.000000  696694.000000
mean       46.390143    -100.500123
std         2.778432      21.840470
min        43.586487    -123.224021
25%        43.697010    -123.110185
50%        43.802264     -79.582106
75%        49.272269     -79.393463
max        49.313349     -79.122044


In [ ]:
# Map 1: Crime Heatmap - Toronto
# Filter Toronto
tor = df[df['city'] == 'Toronto'][['latitude', 'longitude', 'crime_group']].dropna()

# Create map centered on Toronto
m_tor = folium.Map(
    location=[43.7, -79.4],
    zoom_start=11,
    tiles='CartoDB positron'
)

# Heatmap layer
heat_data = tor[['latitude', 'longitude']].values.tolist()
HeatMap(
    heat_data,
    radius = 8,
    blur = 10,
    max_zoom = 13,
    min_opacity=0.3
).add_to(m_tor)

# Title
title_html = ''' 
<h3 align="center" style="font-size:16px; font-family:Arial">
<b>Crime Heatmap - Toronto (2016-2025)</b>
</h3>
'''

m_tor.get_root().html.add_child(folium.Element(title_html))

m_tor.save("../output/maps/toronto_heatmap.html")
print("Saved → toronto_heatmap.html")

Saved → toronto_heatmap.html


In [3]:
# Map 2: Crime Heatmap - Vancouver
van = df[df['city'] == 'Vancouver'][['latitude', 'longitude', 'crime_group']].dropna()

m_van = folium.Map(
    location=[49.27, -123.1],
    zoom_start=12,
    tiles='CartoDB positron'
)

heat_data_van = van[['latitude', 'longitude']].values.tolist()
HeatMap(
    heat_data_van,
    radius = 8,
    blur = 10,
    max_zoom = 13,
    min_opacity=0.3
).add_to(m_van)

title_html_van = '''
<h3 align="center" style="font-size:16px; font-family:Arial">
<b>Crime Heatmap - Vancouver (2016-2025)</b>
</h3>
'''
m_van.get_root().html.add_child(folium.Element(title_html_van))
m_van.save("../output/maps/vancouver_heatmap.html")
print("Saved → vancouver_heatmap.html")

Saved → vancouver_heatmap.html


In [6]:
# Map 3: Crime Type Layer Map
# color mapping for crime groups
crime_colors = {
    'Violent Crime': 'red',
    'Property Crime': 'blue',
    'Property Damage': 'orange',
    'Other': 'gray'
}

def make_layered_map(city, center, zoom):
    city_df = df[df['city'] == city].dropna(subset=['latitude', 'longitude'])

    m = folium.Map(location=center, zoom_start=zoom, tiles='CartoDB positron')

    # Create marker clusters for each crime group
    for group, color in crime_colors.items():
        group_df = city_df[city_df['crime_group'] == group]
        if len(group_df) == 0:
            continue

        fg = folium.FeatureGroup(name=f"{group} ({len(group_df):,})", show = True)

        # Sample max 2000 points per group for performance
        sample = group_df.sample(n=min(2000, len(group_df)), random_state=42)

        for _, row in sample.iterrows():
            folium.CircleMarker(
                location=[row['latitude'], row['longitude']],
                radius=3,
                color=color,
                fill=True,
                fill_opacity=0.5,
                popup=folium.Popup(
                    f"<b>{row['crime_type']}</b><br>"
                    f"Neighbourhood: {row['neighbourhood']}<br>"
                    f"Year: {row['year']} | Hour: {row['hour']}h",
                    max_width=200
                )
            ).add_to(fg)

        fg.add_to(m)
    
    folium.LayerControl(collapsed=False).add_to(m)

    # Title
    title_html = f'''
    <h3 align="center" style="font-size:16px; font-family:Arial">
        <b>Crime by Type — {city} (2016–2025)</b><br>
        <small style="color:gray">Toggle layers to show/hide crime groups</small>
    </h3>
    '''
    m.get_root().add_child(folium.Element(title_html))
    return m

# Toronto
print("Building Toronto map...")
m_tor_layers = make_layered_map(
      'Toronto', [43.7, -79.4], zoom = 11
)
m_tor_layers.save("../output/maps/toronto_crime_layers.html")
print("Saved → toronto_crime_layers.html")

# Vancouver
print("Building Vancouver map...")
m_van_layers = make_layered_map('Vancouver', [49.27, -123.11], zoom = 12)
m_van_layers.save("../output/maps/vancouver_crime_layers.html")
print("Saved → vancouver_crime_layers.html")

Building Toronto map...
Saved → toronto_crime_layers.html
Building Vancouver map...
Saved → vancouver_crime_layers.html


In [18]:
# Map 4: Side-by-side Comparison Map 
tor_sample = df[df['city'] == 'Toronto'][['latitude','longitude']].dropna().sample(5000, random_state=42)
van_sample = df[df['city'] == 'Vancouver'][['latitude','longitude']].dropna().sample(5000, random_state=42)

# Create and save 2 map serately first
m_tor = folium.Map(location=[43.7, -79.4], zoom_start=11, tiles='CartoDB positron')
HeatMap(tor_sample.values.tolist(), radius=8, blur=10).add_to(m_tor)
m_tor.save("../output/maps/toronto_heatmap.html")

m_van = folium.Map(location=[49.27, -123.11], zoom_start=12, tiles='CartoDB positron')
HeatMap(van_sample.values.tolist(), radius=8, blur=10).add_to(m_van)
m_van.save("../output/maps/vancouver_heatmap.html")

# Connect 2 maps into 1 HTML by iframe
html_content = """
<!DOCTYPE html>
<html>
<head>
    <title>Crime Comparison — Toronto vs Vancouver</title>
    <style>
        body { margin: 0; font-family: Arial; background: #f5f5f5; }
        h2 { text-align: center; padding: 15px; color: #333; margin: 0; }
        .container { display: flex; width: 100%; height: 520px; padding: 0 10px 10px; box-sizing: border-box; }
        .map-box { flex: 1; margin: 0 5px; }
        .map-box h3 { text-align: center; margin: 5px 0; font-size: 15px; }
        iframe { width: 100%; height: 480px; border: none; border-radius: 8px; box-shadow: 0 2px 6px rgba(0,0,0,0.15); }
    </style>
</head>
<body>
    <h2>Crime Heatmap Comparison — Toronto vs Vancouver (2016–2025)</h2>
    <div class="container">
        <div class="map-box">
            <h3 style="color:#2196F3">🔵 Toronto</h3>
            <iframe src="toronto_heatmap.html"></iframe>
        </div>
        <div class="map-box">
            <h3 style="color:#FF5722">🔴 Vancouver</h3>
            <iframe src="vancouver_heatmap.html"></iframe>
        </div>
    </div>
</body>
</html>
"""

with open("../output/maps/comparison_map.html", "w") as f:
    f.write(html_content)

print("Saved → comparison_map.html")

Saved → comparison_map.html


In [15]:
# Verify all maps
import os

maps = [
    "../output/maps/toronto_heatmap.html",
    "../output/maps/vancouver_heatmap.html",
    "../output/maps/toronto_crime_layers.html",
    "../output/maps/vancouver_crime_layers.html",
    "../output/maps/comparison_map.html",
]

for path in maps:
    size = os.path.getsize(path) / 1024
    print(f"{'✅' if size > 10 else '❌'} {path.split('/')[-1]:40} {size:,.0f} KB")

✅ toronto_heatmap.html                     13,652 KB
✅ vancouver_heatmap.html                   13,822 KB
✅ toronto_crime_layers.html                4,366 KB
✅ vancouver_crime_layers.html              8,690 KB
✅ comparison_map.html                      402 KB
